# Streaming IMDb Test Set from Disk Through Kafka

This notebook streams a **test set stored on disk** into Kafka using the updated
`src.ingestion.producer` (with the `--data-path` flag), then runs the existing pipeline:

1. Inspect test data on disk (JSONL).
2. Stream test records to Kafka from the notebook.
3. Run the bronze consumer to land data in the bronze layer.
4. (Optional) Run silver and gold jobs to complete the pipeline.


In [2]:
from pathlib import Path
import pandas as pd
import subprocess

# Adjust if you run the notebook from a different directory
PROJECT_ROOT = Path.cwd()
TEST_PATH = PROJECT_ROOT / "data" / "imdb_test.jsonl"  # change if needed

print("Project root:", PROJECT_ROOT)
print("Test file path:", TEST_PATH)


Project root: c:\Users\dyh\Dropbox\Job Search\projects\text-ml-platform
Test file path: c:\Users\dyh\Dropbox\Job Search\projects\text-ml-platform\data\imdb_test.jsonl


In [3]:
if not TEST_PATH.exists():
    raise FileNotFoundError(f"Test file not found at {TEST_PATH}")

df_test = pd.read_json(TEST_PATH, lines=True)
print("Number of test records:", len(df_test))
df_test.head()


FileNotFoundError: Test file not found at c:\Users\dyh\Dropbox\Job Search\projects\text-ml-platform\data\imdb_test.jsonl

## Start Docker services (Kafka, MinIO, etc.)

From a **terminal**, in the project root, ensure the stack is running:

```bash
docker compose -f docker/docker-compose.yml up -d
```

Once services are up, continue with the cells below.


In [ ]:
cmd = [
    "python",
    "-m",
    "src.ingestion.producer",
    "--mode",
    "batch",          # send as fast as possible
    "--data-path",
    str(TEST_PATH),   # stream from local JSONL file
    # "--limit", "1000",  # optional cap on number of records to send
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


In [ ]:
cmd = [
    "python",
    "-m",
    "src.ingestion.bronze_consumer",
    "--batch-size",
    "100",
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


In [ ]:
# Bronze -> Silver (optional)
cmd = [
    "python",
    "-m",
    "src.transformation.silver_job",
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


In [ ]:
# Silver -> Gold embeddings (optional; uses Iceberg when --iceberg is passed)
cmd = [
    "python",
    "-m",
    "src.features.embedding_job",
    "--iceberg",
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)


## Summary

- Streamed a **disk-based test set** (`imdb_test.jsonl`) into Kafka using the updated `producer.py` with `--data-path`.
- Consumed messages into the **bronze layer** via `bronze_consumer`.
- Optionally pushed data through **silver** and **gold** layers using existing jobs.

You can now extend this notebook to run inference with a trained model on the silver/gold test data and write predictions to a dedicated predictions table for evaluation.
